# IEEE-CIS: Historical research benchmark

This notebook preserves the original exploratory random-split code and feature engineering. Outputs are cleared in the public repository. It is not the TiDB model release; see [the release trainer](../src/train_ieee_release.py) and [its validation report](../artifacts/models/ieee_cis_pre_auth_v1_0/validation.json).

In [ ]:
import pandas as pd
from pathlib import Path

# load transaction data with optimized dtypes (CSV folder next to this notebook)
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data/raw/ieee_cis"
train_transaction = pd.read_csv(DATA_DIR / "train_transaction.csv")
test_transaction = pd.read_csv(DATA_DIR / "test_transaction.csv")


In [ ]:
train_identity = pd.read_csv(DATA_DIR / "train_identity.csv")
test_identity = pd.read_csv(DATA_DIR / "test_identity.csv")

In [ ]:
import gc

train_df = train_transaction.merge(train_identity, on='TransactionID', how='left')
test_df = test_transaction.merge(test_identity, on='TransactionID', how='left')

del train_transaction, train_identity, test_transaction, test_identity
gc.collect()

print(f"Train shape: {train_df.shape}")
print(f"Test shape: {test_df.shape}")

Exploratory Data Analysis

In [ ]:
print("Dataset Shape: ", train_df.shape)


In [ ]:
print("\nDataset Info: \n", train_df.info())


In [ ]:
print("\nMissing Values: \n", train_df.isnull().sum())


In [ ]:
print("\nDescriptive Statistics: \n", train_df.describe().T)


In [ ]:
print("Dataset Columns: \n", train_df.columns)


In [ ]:
import matplotlib.pyplot as plt

plt.pie(train_df['isFraud'].value_counts(), labels=['Not Fraud', 'Fraud'], autopct='%0.2f%%')
plt.title('Class Distribution')
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
plt.hist(train_df['TransactionDT'], label='Train', bins = 30, color='green')
plt.hist(test_df['TransactionDT'], label='Test', bins = 30, color= 'blue')
plt.ylabel('Count')
plt.title('Transaction Date Time')
plt.legend()
plt.show()

In [ ]:
import seaborn as sns

fig, ax = plt.subplots(2, 1, figsize=(8, 10))

sns.histplot(data=train_df, x='TransactionAmt', hue='isFraud', kde=True, palette='viridis',log_scale=True , ax=ax[0])
ax[0].set(title='Distribution of Transaction Amount (Train)', xlabel='Transaction Amount(Log Scale)', ylabel='Count')
ax[0].legend(title='isFraud', labels=['Fraud', 'Not Fraud'])

sns.histplot(data=test_df, x='TransactionAmt', kde=True,log_scale=True , color='skyblue', ax=ax[1])
ax[1].set(title='Distribution of Transaction Amount (Test)', xlabel='Transaction Amount(Log Scale)', ylabel='Count')

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np

fig, ax = plt.subplots(2, 1, figsize=(8, 10))

train_df['TransactionAmt_log'] = np.log1p(train_df['TransactionAmt'])
sns.histplot(data=train_df, x='TransactionAmt_log', hue='isFraud', kde=True, palette='viridis', ax=ax[0])
ax[0].set(title='Distribution of Log of Transaction Amount (Train)', xlabel='Transaction Amount', ylabel='Count')
ax[0].legend(title='isFraud', labels=['Fraud', 'Not Fraud'])

test_df['TransactionAmt_log'] = np.log1p(test_df['TransactionAmt'])
sns.histplot(data=test_df, x='TransactionAmt_log', kde=True, color='skyblue', ax=ax[1])
ax[1].set(title='Distribution of Log of Transaction Amount (Test)', xlabel='Transaction Amount', ylabel='Count')

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(8, 10))

sns.countplot(data=train_df, x="ProductCD", ax=ax[0], hue = "isFraud")
ax[0].set_title('ProductCD (Train)')
ax[0].legend(title='isFraud', labels=['Fraud', 'Not Fraud'])

sns.countplot(data=test_df, x="ProductCD", ax=ax[1])
ax[1].set_title('ProductCD (Test)')
ax[1].legend(title='isFraud', labels=['Fraud', 'Not Fraud'])

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(8, 10))

sns.countplot(data=train_df, x='card4', hue='isFraud', ax=ax[0])
ax[0].set_title('Distribution of Card 4 (Train)')
ax[0].set_xlabel('Card 4')
ax[0].set_ylabel('Count')
ax[0].legend(title='isFraud', labels=['Not Fraud', 'Fraud'])

sns.countplot(data=test_df, x='card4', ax=ax[1])
ax[1].set_title('Distribution of Card 4 (Test)')
ax[1].set_xlabel('Card 4')
ax[1].set_ylabel('Count')

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(8, 10))


sns.countplot(data=train_df, x='card6', hue='isFraud', ax=ax[0])
ax[0].set_title('Distribution of Card 6 (Train)')
ax[0].set_xlabel('Card 6')
ax[0].set_ylabel('Count')
ax[0].legend(title='isFraud', labels=['Not Fraud', 'Fraud'])

sns.countplot(data=test_df, x='card6', ax=ax[1])
ax[1].set_title('Distribution of Card 6 (Test)')
ax[1].set_xlabel('Card 6')
ax[1].set_ylabel('Count')

plt.tight_layout()
plt.show()

Feature Engineering

In [ ]:
def engineer_features_optimized(df):
    """Memory efficient feature engineering"""
    
    # create unique ID for aggregations
    df['uid'] = df['card1'].astype(str) + '_' + df['card2'].fillna(0).astype(str)
    
    # transaction amount features
    df['TransactionAmt_log'] = np.log1p(df['TransactionAmt'])
    df['TransactionAmt_decimal'] = (df['TransactionAmt'] - df['TransactionAmt'].astype(int))
    
    # time-based features
    df['TransactionDT_day'] = (df['TransactionDT'] // (24*3600)).astype(np.int16)
    df['TransactionDT_hour'] = ((df['TransactionDT'] % (24*3600)) // 3600).astype(np.int8)
    
    # aggregation features (memory-efficient)
    for agg_col in ['TransactionAmt']:
        for stat in ['mean', 'std']:
            col_name = f'uid_{agg_col.lower()}_{stat}'
            df[col_name] = df.groupby('uid')[agg_col].transform(stat)
            df[col_name] = df[col_name].astype(np.float32)
    
    # transaction amount deviation
    df['TransactionAmt_deviation'] = df['TransactionAmt'] / (df['uid_transactionamt_mean'] + 1e-5)
    df['TransactionAmt_deviation'] = df['TransactionAmt_deviation'].astype(np.float32)
    
    # frequency encoding for top features
    for col in ['card1', 'card2', 'addr1']:
        freq = df[col].value_counts(dropna=False)
        df[f'{col}_freq'] = df[col].map(freq).astype(np.float32)
    
    # browser extraction from id_31
    if 'id_31' in df.columns:
        df['browser'] = df['id_31'].astype(str).str.extract('([a-zA-Z]+)', expand=False).fillna('unknown')
        df['browser'] = df['browser'].astype('category')
    
    # drop temporary columns
    df.drop(['uid'], axis=1, inplace=True)
    
    return df


In [ ]:
# apply feature engineering
train_df = engineer_features_optimized(train_df)
test_df = engineer_features_optimized(test_df)

gc.collect()
print(f"Train shape after feature engineering: {train_df.shape}")
print(f"Test shape after feature engineering: {test_df.shape}")

Data Cleaning


In [ ]:
# remove duplicates
print(f"Train shape before dropping duplicates: {train_df.shape}")
train_df.drop_duplicates(inplace=True)
print(f"Train shape after dropping duplicates: {train_df.shape}")

print(f"Test shape before dropping duplicates: {test_df.shape}")
test_df.drop_duplicates(inplace=True)
print(f"Test shape after dropping duplicates: {test_df.shape}")

In [ ]:
# drop columns with >80% missing values
threshold = 0.8

print(f"Train shape before dropping columns: {train_df.shape}")
train_missing = train_df.isnull().sum() / len(train_df)
cols_to_drop = train_missing[train_missing > threshold].index.tolist()
train_df.drop(columns=cols_to_drop, inplace=True)
print(f"Dropped {len(cols_to_drop)} columns with >{threshold*100}% missing")
print(f"Train shape after: {train_df.shape}")

print(f"Test shape before: {test_df.shape}")
test_missing = test_df.isnull().sum() / len(test_df)
cols_to_drop_test = test_missing[test_missing > threshold].index.tolist()
test_df.drop(columns=cols_to_drop_test, inplace=True)
print(f"Dropped {len(cols_to_drop_test)} columns with >{threshold*100}% missing")
print(f"Test shape after: {test_df.shape}")

Prepare Data for Modeling

In [ ]:
target = 'isFraud'
y = train_df[target]
X_train = train_df.drop(columns=[target, 'TransactionID'])
X_test = test_df.drop(columns=['TransactionID'])

print(f"Features shape: {X_train.shape}")
print(f"Target distribution:\n{y.value_counts()}")
print(f"Fraud rate: {y.mean()*100:.2f}%")

In [ ]:
from sklearn.preprocessing import OrdinalEncoder
import gc

# identify categorical columns
cat_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()
test_cat_cols = X_test.select_dtypes(include=['object', 'category']).columns.tolist()

# find common columns
common_cat_cols = list(set(cat_cols) & set(test_cat_cols))
print(f"Encoding {len(common_cat_cols)} categorical columns")

# initialise ordinal encoder
oe = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)

# handle missing values and encode
X_train[common_cat_cols] = X_train[common_cat_cols].astype(str)
X_test[common_cat_cols] = X_test[common_cat_cols].astype(str)

# apply encoding
X_train[common_cat_cols] = oe.fit_transform(X_train[common_cat_cols])
X_test[common_cat_cols] = oe.transform(X_test[common_cat_cols])

# optimize memory
num_cols = X_train.select_dtypes(include=['float64']).columns
for col in num_cols:
    X_train[col] = X_train[col].astype('float32')
    if col in X_test.columns:
        X_test[col] = X_test[col].astype('float32')

print(f"Final feature count: {X_train.shape[1]}")

# identify leftover object columns
non_numeric_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()

# drop columns
print(f"Dropping {len(non_numeric_cols)} unencoded columns")
X_train = X_train.drop(columns=non_numeric_cols)
X_test = X_test.drop(columns=non_numeric_cols, errors='ignore')

# reindex adds missing columns to test and fills them with 0
X = X_train
X_test = X_test.reindex(columns=X.columns, fill_value=0)

# free memory
gc.collect()

Model Training with Cross Validation

In [ ]:
from sklearn.model_selection import train_test_split

X_tr, X_val, y_tr, y_val = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print(X_tr.shape, X_val.shape)

In [ ]:
import sys
import subprocess

# Ensure xgboost exists in the currently running notebook environment
try:
    from xgboost import XGBClassifier
except ModuleNotFoundError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "xgboost"])
    from xgboost import XGBClassifier

scale = (y_tr == 0).sum() / (y_tr == 1).sum()

# Use GPU when available; fallback to CPU if CUDA runtime is unavailable
try:
    model = XGBClassifier(
        n_estimators=400,
        max_depth=8,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=scale,
        objective="binary:logistic",
        eval_metric="auc",
        tree_method="hist",
        device="cuda",
        random_state=42,
    )
    model.fit(
        X_tr,
        y_tr,
        eval_set=[(X_val, y_val)],
        verbose=50,
    )
except Exception:
    model = XGBClassifier(
        n_estimators=400,
        max_depth=8,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=scale,
        objective="binary:logistic",
        eval_metric="auc",
        tree_method="hist",
        device="cpu",
        random_state=42,
    )
    model.fit(
        X_tr,
        y_tr,
        eval_set=[(X_val, y_val)],
        verbose=50,
    )

In [ ]:
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

y_pred = model.predict(X_val)
y_prob = model.predict_proba(X_val)[:, 1]

print("ROC AUC:", roc_auc_score(y_val, y_prob))
print(confusion_matrix(y_val, y_pred))
print(classification_report(y_val, y_pred))

In [ ]:
import matplotlib.pyplot as plt
from xgboost import plot_importance

plt.figure(figsize=(10,8))
plot_importance(model, max_num_features=20)
plt.show()

In [ ]:
# Dashboard linkage: export validation predictions for src/dashboard/dashboard_app.py
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
ARTIFACT_DIR = PROJECT_ROOT / "artifacts/dashboard"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

val_export = pd.DataFrame({
    "y_true": y_val.values if hasattr(y_val, "values") else y_val,
    "y_pred": y_pred,
    "risk_score": y_prob,
})

# X_val.index is the row label in the training matrix, not the business TransactionID.
# Align with the same rows in train_df so the dashboard can merge on TransactionID.
val_export["TransactionID"] = train_df.loc[X_val.index, "TransactionID"].values

feature_output_path = ARTIFACT_DIR / "cleaned_ieee_features.csv"
if "isFraud" in X.columns:
    X.drop(columns=["isFraud"], inplace=True)
X["isFraud"] = y.loc[X.index].values if hasattr(y, "loc") else y
X.to_csv(feature_output_path, index=False)
X.drop(columns=["isFraud"], inplace=True)
print("Saved:", feature_output_path)

output_path = ARTIFACT_DIR / "ieee_cis_val_predictions.csv"
val_export.to_csv(output_path, index=False)
print("Saved:", output_path)

# Run dashboard from terminal:
# streamlit run src/dashboard/dashboard_app.py
